# Python Enumerations: A Second Advanced Problem-Solving Tutorial

## New problems, incremental solutions, detailed explanations, and executable checks

This is a **new tutorial-style set** rather than a reshuffling of the previous 28-problem workbook. It follows the supplied lesson's small-step pattern: introduce a question, test the relevant behavior, explain the consequence, implement a solution in successive cells, and verify it.

**Topic:** Python `enum`: member identity, names and values, value/name lookup, iteration, class boundaries, inheritance, and practical use in parsers and data systems. The exercises extend those foundations to dynamic declarations, safe input handling, CSV/SQLite/binary data, an expression interpreter, asynchronous orchestration, and an integrated capstone.

**Prerequisites:** Python 3.10+ and Jupyter. Only Python's standard library is used. Run **Kernel → Restart & Run All** from top to bottom. All exception demonstrations catch the intended exception, so a normal run contains no uncaught errors.

**Teaching convention:** each problem has a statement, smaller steps with separate explanation and code cells, a finished solution/check, an edge-case review, and an optional challenge. Predicted outcomes are explicitly checked using `assert` rather than being silently assumed.

## Road map and best practices

1. **Reason about type identity before values.** An enum member is an instance of *one particular enum class*, not simply an integer or string.
2. **Distinguish a member's name from its value.** If a field is a name, use `EnumClass[name]`; if it is a value, use `EnumClass(value)`.
3. **Validate at boundaries.** External text, database rows, and byte sequences are not guaranteed to be correct just because the internal model uses enums.
4. **Do not couple persisted formats to declaration order.** Publish explicit names/codes and define migrations where a contract changes.
5. **Keep solutions observable and testable.** Each exercise includes normal cases and at least one failure or edge case; integration tests come last.

The exercises use fresh class names per section so that all cells work in one kernel and can be revisited independently after running earlier setup cells. Exercises 1–7 emphasize enum semantics; exercises 8–16 develop full workflows.

## Shared setup: visible checks and deliberate failures

The small helper below verifies that an expected exception really occurs. It does not swallow an unrelated exception: that makes it useful when learning why one kind of lookup raises `KeyError` while another raises `ValueError`. No third-party test packages are required.

In [1]:
from enum import Enum, auto
from collections import Counter
from dataclasses import dataclass


def must_raise(expected_type, operation):
    """Return the caught error, or fail if the expected error is absent."""
    try:
        operation()
    except expected_type as error:
        return error
    raise AssertionError(f"Expected {expected_type.__name__}")

assert must_raise(ValueError, lambda: int("not-a-number"))
print("Setup is ready.")

Setup is ready.


## Problem 01 — Identity across independently defined enum classes

**Challenge.** A hot-reload system defines a new enum class with the same member names and values as an old class. Explain why the old and new members cannot be treated as interchangeable.

**Success criteria.** Demonstrate equal underlying values but distinct enum identities; implement an explicit value-based conversion and reject unknown codes.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Form a hypothesis about names, values, and classes

Create two independent `Enum` subclasses that look identical. An enum member is bound to its *defining class*: identical `.name` and `.value` data does not make it the same member. Start by checking each attribute separately.

In [2]:
class OldPhase01(Enum):
    READY = "ready"
    BUSY = "busy"

class NewPhase01(Enum):
    READY = "ready"
    BUSY = "busy"

old_member = OldPhase01.READY
new_member = NewPhase01.READY
assert old_member.name == new_member.name
assert old_member.value == new_member.value
assert old_member is not new_member
assert old_member != new_member
print(old_member, new_member, old_member.value)

OldPhase01.READY NewPhase01.READY ready


### Step 2 — Observe strict lookup by class

`EnumClass(value)` creates no new member: it returns an existing member of that specific enum class. A value lookup is useful at a migration boundary. Supplying an old member object to a different class is not a portable conversion policy; supply its serialized value explicitly.

In [3]:
assert OldPhase01("ready") is OldPhase01.READY
assert NewPhase01(old_member.value) is NewPhase01.READY
assert isinstance(old_member, OldPhase01)
assert not isinstance(old_member, NewPhase01)
assert must_raise(ValueError, lambda: NewPhase01("missing"))

### Step 3 — Write an intentionally explicit adapter

Adapters should say which representation they accept. The function below accepts *only* the old enum class; it serializes to the old value, then validates against the new class. This ensures a caller cannot pass an arbitrary string and accidentally skip the intended type boundary.

In [4]:
def upgrade_phase01(source: OldPhase01) -> NewPhase01:
    if not isinstance(source, OldPhase01):
        raise TypeError("expected an OldPhase01 member")
    return NewPhase01(source.value)

assert upgrade_phase01(OldPhase01.BUSY) is NewPhase01.BUSY
assert must_raise(TypeError, lambda: upgrade_phase01("busy"))
assert must_raise(TypeError, lambda: upgrade_phase01(NewPhase01.BUSY))

### Step 4 — Verify a failure in a changed schema

A migration can fail even if the old payload is legitimate: the new schema might have retired an old code. This is a *schema compatibility* problem, not a reason to compare enum members by their printed representation.

In [5]:
class RetiredPhase01(Enum):
    READY = "ready"

assert must_raise(ValueError, lambda: RetiredPhase01(OldPhase01.BUSY.value))
print("Migration refuses a retired value.")

Migration refuses a retired value.


### Worked-solution review

**Why it works.** Enum equality and membership reflect class identity; converting through a documented external value crosses class boundaries without pretending the classes are identical.

**Important boundary / common mistake.** Avoid testing `str(member)` or `repr(member)` as a serialization contract; those are presentation-oriented representations.

**Try it yourself:** Add a mapping from the retired `"busy"` value to a replacement state, and explain whether the translation loses meaning.

In [6]:
assert True
print("Problem 01: all assertions passed")

Problem 01: all assertions passed


## Problem 02 — Construct an enum from a trusted schema

**Challenge.** An application loads a *trusted* configuration declaring integer protocol codes. Construct the enum dynamically while protecting against duplicate names, duplicate codes, and invalid public identifiers.

**Success criteria.** Validate the schema first; build members with the functional API; show both name and value lookup and distinguish schema errors from lookup errors.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Examine the dynamic construction API

The functional form `Enum("Name", mapping)` creates a new enum class. The input schema is data, not executable Python source. Never use `eval` to turn strings into enum declarations.

In [7]:
raw_codes02 = [("START", 10), ("STOP", 20), ("PAUSE", 30)]
QuickProtocol02 = Enum("QuickProtocol02", dict(raw_codes02))
assert QuickProtocol02["START"].value == 10
assert QuickProtocol02(20) is QuickProtocol02.STOP
print([(member.name, member.value) for member in QuickProtocol02])

[('START', 10), ('STOP', 20), ('PAUSE', 30)]


### Step 2 — Validate *before* calling the constructor

Converting a sequence of name–code pairs directly to a dictionary would silently overwrite repeated names. We must detect duplicates while the information is still available. We also require uppercase ASCII identifiers and exact integer codes (rather than booleans, which are subclasses of `int`).

In [8]:
import re

_IDENTIFIER02 = re.compile(r"[A-Z][A-Z0-9_]*\Z")

def validate_schema02(pairs):
    pairs = list(pairs)
    if not pairs:
        raise ValueError("a protocol must define at least one code")
    names, codes = set(), set()
    for name, code in pairs:
        if not isinstance(name, str) or not _IDENTIFIER02.fullmatch(name):
            raise ValueError(f"invalid name: {name!r}")
        if name in names:
            raise ValueError(f"duplicate name: {name}")
        if type(code) is not int or not 0 <= code <= 255:
            raise ValueError(f"invalid uint8 code: {code!r}")
        if code in codes:
            raise ValueError(f"duplicate code: {code}")
        names.add(name)
        codes.add(code)
    return dict(pairs)

### Step 3 — Build only after successful validation

Now expose one small construction function. `module=__name__` tells Python where the dynamic class is defined and can aid discoverability; reliably pickling dynamic classes also requires a stable module-level binding, which we do not promise in this example.

In [9]:
def make_protocol02(class_name, pairs):
    if not isinstance(class_name, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", class_name):
        raise ValueError("invalid class name")
    return Enum(class_name, validate_schema02(pairs), module=__name__)

Protocol02 = make_protocol02("Protocol02", raw_codes02)
assert Protocol02.START.value == 10
assert Protocol02["STOP"] is Protocol02(20)
assert list(Protocol02.__members__) == ["START", "STOP", "PAUSE"]

### Step 4 — Test the malicious-looking and accidental inputs

An apparently minor duplicate code would normally create an alias, but this protocol requires one code per label. Reject it before enum construction. There is no need to catch errors with `except Exception` because the expected failures are known.

In [10]:
bad_schemas02 = [
    [("START", 1), ("START", 2)],
    [("START", 1), ("STOP", 1)],
    [("start", 1)],
    [("START", True)],
    [("START", -1)],
    [],
]
for schema in bad_schemas02:
    must_raise(ValueError, lambda schema=schema: make_protocol02("Bad02", schema))
assert must_raise(KeyError, lambda: Protocol02["MISSING"])
assert must_raise(ValueError, lambda: Protocol02(99))

### Worked-solution review

**Why it works.** Separate schema validation from class creation; preserve duplicate information until it has been checked. The resulting enum offers stable, explicit names and byte-sized codes.

**Important boundary / common mistake.** Only use this pattern for trusted schema data; if a schema changes during a deployment, version the external contract. A dynamic class constructed twice is still two different classes.

**Cost.** Validation and construction use O(n) expected time and O(n) auxiliary space for n schema entries.

**Try it yourself:** Add an optional prohibition against reserved names such as `name` and `value`, and document your chosen naming policy.

In [11]:
assert True
print("Problem 02: all assertions passed")

Problem 02: all assertions passed


## Problem 03 — Build a collision-safe, human-friendly member-name parser

**Challenge.** A UI accepts labels such as `in progress`, `IN-PROGRESS`, and `In_Progress`. Map labels to canonical enum members, but detect ambiguous normalization rather than silently picking a member.

**Success criteria.** Normalize input, build a checked lookup, keep the original member names intact, and reject unknown or ambiguous labels.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Separate external spelling from internal names

Names like `IN_PROGRESS` are stable programming identifiers. UI strings have looser conventions. A normalization function can bridge the two, but normalization is not necessarily one-to-one. We will prove that before trusting it.

In [12]:
class JobStage03(Enum):
    NEW = "N"
    IN_PROGRESS = "P"
    DONE = "D"


def normalize_label03(label):
    if not isinstance(label, str):
        raise TypeError("label must be text")
    return "_".join(label.strip().replace("-", " ").replace("_", " ").upper().split())

assert normalize_label03(" in-progress ") == "IN_PROGRESS"
assert normalize_label03("In_Progress") == "IN_PROGRESS"

### Step 2 — Construct a checked index

The parser operates on declared names, not enum values. If two different names collapse to the same normalized label, building the index fails immediately rather than giving different answers depending on declaration order.

In [13]:
def name_index03(enum_type, normalizer=normalize_label03):
    lookup = {}
    for name, member in enum_type.__members__.items():
        key = normalizer(name)
        if key in lookup and lookup[key] is not member:
            raise ValueError(f"ambiguous normalized label: {key}")
        lookup[key] = member
    return lookup

INDEX03 = name_index03(JobStage03)
assert INDEX03["IN_PROGRESS"] is JobStage03.IN_PROGRESS

### Step 3 — Parse without dynamic attribute access

Use a precomputed dictionary and raise a domain-specific error for invalid strings. `getattr(EnumClass, user_input)` is less explicit, and it can hit non-member attributes such as `__members__`.

In [14]:
def parse_stage03(label):
    key = normalize_label03(label)
    try:
        return INDEX03[key]
    except KeyError as exc:
        raise ValueError(f"unknown job stage: {label!r}") from exc

assert parse_stage03(" in-progress ") is JobStage03.IN_PROGRESS
assert parse_stage03("done") is JobStage03.DONE
assert must_raise(ValueError, lambda: parse_stage03("almost done"))
assert must_raise(TypeError, lambda: parse_stage03(None))

### Step 4 — Prove collision detection rather than assuming it works

Python member names can differ in ways the UI normalizer erases. Check an intentionally problematic enum class. If aliases point to the *same member*, the index permits them; the ambiguity we must reject is two different members with the same normalized label.

In [15]:
class Collision03(Enum):
    A_B = 1
    A__B = 2

assert must_raise(ValueError, lambda: name_index03(Collision03))
assert JobStage03["IN_PROGRESS"] is parse_stage03("in progress")
print("The public name, normalized UI label, and wire value remain distinct.")

The public name, normalized UI label, and wire value remain distinct.


### Worked-solution review

**Why it works.** Normalize only at the boundary; index actual enum members and detect collisions when constructing the index. This protects stable internal names while accepting convenient input spellings.

**Important boundary / common mistake.** Do not normalize serialized protocol values unless the protocol explicitly permits it. Names, display labels, and external codes may each have different compatibility requirements.

**Cost.** Build the index in O(n); each subsequent lookup takes O(1) expected time.

**Try it yourself:** Expose all accepted labels to a UI and add a test for non-ASCII capitalization policies.

In [16]:
assert True
print("Problem 03: all assertions passed")

Problem 03: all assertions passed


## Problem 04 — Reuse behavior safely with a member-free mixin

**Challenge.** Two unrelated enums need the same method for formatting audit messages. Share behavior without attempting to extend an enum that already defines members.

**Success criteria.** Implement a mixin, demonstrate two distinct enum classes, and show why a subclass cannot add members to a populated parent enum.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Distinguish methods from members

Python permits inheriting reusable behavior from a class that has no enum members. A plain mixin placed **before** `Enum` in the base-class list supplies methods but does not create enum entries.

In [17]:
class AuditLabel04:
    def audit_label(self):
        return f"{type(self).__name__}.{self.name}={self.value}"

class Payment04(AuditLabel04, Enum):
    REQUESTED = "requested"
    SETTLED = "settled"

class Shipment04(AuditLabel04, Enum):
    PACKED = "packed"
    SENT = "sent"

assert Payment04.SETTLED.audit_label() == "Payment04.SETTLED=settled"
assert Shipment04.SENT.audit_label() == "Shipment04.SENT=sent"

### Step 2 — Also use an empty enum base when it provides genuine common behavior

An enum base with *no members* can be subclassed. Contrast that with subclassing `Payment04`, which already has members. An enum class cannot be extended with additional members after it has defined any.

In [18]:
class CodedBase04(Enum):
    def code_tag(self):
        return f"code:{self.value}"

class Signal04(CodedBase04):
    GREEN = 1
    RED = 2

assert Signal04.GREEN.code_tag() == "code:1"

def attempt_extend04():
    class MorePayments04(Payment04):
        REFUNDED = "refunded"
    return MorePayments04

must_raise(TypeError, attempt_extend04)

TypeError("<enum 'MorePayments04'> cannot extend <enum 'Payment04'>")

### Step 3 — Check that common behavior does not imply common identity

Do not interpret a shared mixin as a shared domain. Methods can be reused; enum members remain different types. At an API boundary, check for exactly the domain you expect.

In [19]:
def payment_audit04(value: Payment04):
    if not isinstance(value, Payment04):
        raise TypeError("payment event required")
    return value.audit_label()

assert payment_audit04(Payment04.REQUESTED) == "Payment04.REQUESTED=requested"
assert must_raise(TypeError, lambda: payment_audit04(Shipment04.PACKED))
assert len(list(Payment04)) == 2

### Worked-solution review

**Why it works.** Place shared behavior in a member-free mixin or empty enum base. Actual members belong to their concrete enum, so inheriting a method does not leak one domain’s values into another.

**Important boundary / common mistake.** Do not put enum members in a base class that you intend to extend, and do not confuse method inheritance with conversion between domain types.

**Try it yourself:** Add a `.short_label()` method to `AuditLabel04` and use it from both concrete enums.

In [20]:
assert True
print("Problem 04: all assertions passed")

Problem 04: all assertions passed


## Problem 05 — Design deterministic auto-generated string codes

**Challenge.** You want enum values that exactly match stable lowercase protocol identifiers without manually duplicating every spelling. Implement and test a custom `auto()` generation rule.

**Success criteria.** Implement `_generate_next_value_`, show the names-to-values mapping, and explain the stability limitation if names later change.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Define the generation rule before any members

`auto()` delegates value generation to `_generate_next_value_`. Defining the hook at the top of the enum body makes the intent explicit. Unlike default auto-numbering, generated codes below depend on the declared member **name**.

In [21]:
class ServiceAction05(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower().replace("_", "-")

    CREATE_USER = auto()
    DELETE_USER = auto()
    READ_AUDIT = auto()

assert ServiceAction05.CREATE_USER.value == "create-user"
assert ServiceAction05.READ_AUDIT.value == "read-audit"
print([(x.name, x.value) for x in ServiceAction05])

[('CREATE_USER', 'create-user'), ('DELETE_USER', 'delete-user'), ('READ_AUDIT', 'read-audit')]


### Step 2 — Round-trip a code, not a generated position

The public value may travel through JSON or a queue. Its uniqueness is visible in the enum, and the caller uses the value constructor to decode it. There is no comparison with the declaration position.

In [22]:
def encode_action05(action: ServiceAction05) -> str:
    if not isinstance(action, ServiceAction05):
        raise TypeError("action must be ServiceAction05")
    return action.value

def decode_action05(code: str) -> ServiceAction05:
    if not isinstance(code, str):
        raise TypeError("code must be text")
    return ServiceAction05(code)

for action in ServiceAction05:
    assert decode_action05(encode_action05(action)) is action
assert must_raise(ValueError, lambda: decode_action05("unknown"))

### Step 3 — Explore a schema evolution hazard

The rule is deterministic but not inherently immutable: renaming an enum member changes its generated code. For long-lived external contracts, pin explicit values or add a migration table. Here we demonstrate the difference without changing the original enum.

In [23]:
class RenamedAction05(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower().replace("_", "-")

    ADD_USER = auto()

assert RenamedAction05.ADD_USER.value != ServiceAction05.CREATE_USER.value
print("Renaming a member changes generated wire codes: publish with care.")

Renaming a member changes generated wire codes: publish with care.


### Worked-solution review

**Why it works.** `_generate_next_value_` is called during class construction and can generate readable, deterministic codes. The decoder still validates through the enum class.

**Important boundary / common mistake.** This is suitable only when member names themselves are treated as part of a controlled compatibility contract. Avoid changing names after values have been persisted.

**Try it yourself:** Add a collision check if your transformation removes distinctions such as underscore and hyphen from generated values.

In [24]:
assert True
print("Problem 05: all assertions passed")

Problem 05: all assertions passed


## Problem 06 — Understand equality-based aliases with unhashable member values

**Challenge.** The business requirement is “two entries with equal list payloads must refer to the same enum member.” Explore why this is a dangerous implicit schema rule and replace the list values with stable identifiers.

**Success criteria.** Show aliasing caused by equal list values; show that the members remain hashable; refactor to immutable explicit codes plus separate metadata.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Observe how equal values can create an alias

The enum machinery identifies duplicate values by equality. Lists are unhashable, but they are still comparable; two different-looking declarations with equal list contents become aliases. This example intentionally shows the questionable design.

In [25]:
class FragileRoute06(Enum):
    PRIMARY = ["read", "write"]
    BACKUP = ["read", "write"]
    OBSERVER = ["read"]

assert FragileRoute06.BACKUP is FragileRoute06.PRIMARY
assert len(list(FragileRoute06)) == 2
assert list(FragileRoute06.__members__) == ["PRIMARY", "BACKUP", "OBSERVER"]
assert isinstance(hash(FragileRoute06.PRIMARY), int)

### Step 2 — Separate member identity from mutable metadata

A member is usable as a dictionary key even if its `.value` is unhashable. But the list held in `.value` can be mutated, which can change what consumers observe. Do not use mutable payloads as public identifiers.

In [26]:
route_lookup06 = {FragileRoute06.PRIMARY: "router-A"}
assert route_lookup06[FragileRoute06.BACKUP] == "router-A"
assert must_raise(TypeError, lambda: hash(FragileRoute06.PRIMARY.value))
print("Alias names identify the SAME enum member.")

Alias names identify the SAME enum member.


### Step 3 — Refactor to explicit stable identifiers and a separate capability table

Keep enum values simple and unique. A `frozenset` in a separate mapping represents capabilities without making the members aliases merely because two capability sets are equal. This separates *identity* from *configuration*.

In [27]:
class Route06(Enum):
    PRIMARY = "primary"
    BACKUP = "backup"
    OBSERVER = "observer"

CAPABILITIES06 = {
    Route06.PRIMARY: frozenset({"read", "write"}),
    Route06.BACKUP: frozenset({"read", "write"}),
    Route06.OBSERVER: frozenset({"read"}),
}
assert Route06.PRIMARY is not Route06.BACKUP
assert CAPABILITIES06[Route06.PRIMARY] == CAPABILITIES06[Route06.BACKUP]
assert len(list(Route06)) == 3
assert all(isinstance(x, frozenset) for x in CAPABILITIES06.values())

### Worked-solution review

**Why it works.** Equality of associated values affects aliasing; two business concepts may require distinct enum identities even if their metadata matches. Unique immutable codes plus separate metadata express that intent.

**Important boundary / common mistake.** Do not infer the number of declared names from `len(EnumClass)` when aliases exist: iteration omits aliases, whereas `__members__` includes them.

**Try it yourself:** Add a check ensuring every `Route06` member has an entry in the capability table.

In [28]:
assert True
print("Problem 06: all assertions passed")

Problem 06: all assertions passed


## Problem 07 — Make structural pattern matching genuinely type-aware

**Challenge.** A message router receives a mixture of enum members, strings, integers, and `None`. Use `match` safely: enum constants should be matched as qualified value patterns, and raw inputs should not accidentally hit a wildcard.

**Success criteria.** Show a genuine enum pattern match, add type validation, and reject unknown strings without permissive fall-through.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Understand qualified enum patterns

A qualified name such as `Channel07.EMAIL` is a constant value pattern. A bare name such as `EMAIL` is a *capture pattern*, not a reference to the enum member. We therefore always qualify enum members in a `case` clause.

In [29]:
class Channel07(Enum):
    EMAIL = "email"
    SMS = "sms"
    PUSH = "push"


def route_member07(channel: Channel07):
    if not isinstance(channel, Channel07):
        raise TypeError("expected a Channel07 member")
    match channel:
        case Channel07.EMAIL:
            return "smtp"
        case Channel07.SMS:
            return "sms-gateway"
        case Channel07.PUSH:
            return "push-service"
        case _:
            raise AssertionError("update route_member07 for new enum members")

assert route_member07(Channel07.EMAIL) == "smtp"
assert route_member07(Channel07.PUSH) == "push-service"

### Step 2 — Parse external values at the boundary

`Channel07("email")` is value lookup. The router then receives only real members. This makes the match statement easier to reason about than mixing strings and members in the same branches.

In [30]:
def route_external07(raw):
    if type(raw) is not str:
        raise TypeError("channel code must be a string")
    return route_member07(Channel07(raw))

assert route_external07("sms") == "sms-gateway"
assert must_raise(ValueError, lambda: route_external07("fax"))
assert must_raise(TypeError, lambda: route_external07(None))

### Step 3 — Demonstrate that adding a member must update routing

Exhaustiveness is a maintenance responsibility in runtime Python. We can test coverage by building a handler table with exactly the same set of enum keys; missing and extra keys become a detectable test failure.

In [31]:
HANDLERS07 = {
    Channel07.EMAIL: "smtp",
    Channel07.SMS: "sms-gateway",
    Channel07.PUSH: "push-service",
}
assert set(HANDLERS07) == set(Channel07)
assert all(HANDLERS07[x] == route_member07(x) for x in Channel07)
print("Pattern matching and table-based coverage agree.")

Pattern matching and table-based coverage agree.


### Worked-solution review

**Why it works.** Type-check external data, decode it with the right enum constructor, and use qualified constants in pattern matching. A coverage check guards future enum expansions.

**Important boundary / common mistake.** Never write `case EMAIL:` intending to match a constant: an unqualified bare name captures any subject. `case _:` should report an unexpected member rather than silently accept it.

**Try it yourself:** Add `Channel07.WEBHOOK` to a fresh copy of the class and show the coverage assertion exposing a missing handler.

In [32]:
assert True
print("Problem 07: all assertions passed")

Problem 07: all assertions passed


## Problem 08 — Collect precise errors when parsing mixed enum fields

**Challenge.** An inbound API request has several independently validated fields. Instead of stopping at the first bad value, report field-specific diagnostics while preserving exact enum types for successful fields.

**Success criteria.** Build an explicit field schema; return typed results only for a fully valid record; handle missing, wrong-type, and unknown-code cases separately.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Model independent domains with independent enum classes

Two fields may contain similar-looking strings but represent separate concepts. Do not replace them with a single generic string enum. Define each domain and an explicit mapping from field names to domain classes.

In [33]:
class Region08(Enum):
    EU = "eu"
    US = "us"

class Tier08(Enum):
    FREE = "free"
    PRO = "pro"

class Mode08(Enum):
    FAST = "fast"
    SAFE = "safe"

SCHEMA08 = {"region": Region08, "tier": Tier08, "mode": Mode08}
assert len(SCHEMA08) == 3

### Step 2 — Design a diagnostic result instead of printing errors

Use a small frozen dataclass as a structured error. A parser should return useful information to its caller, not print user-data errors in arbitrary notebook output. In this exercise a successful result and a list of diagnostics are returned together.

In [34]:
@dataclass(frozen=True)
class Diagnostic08:
    field: str
    problem: str


def parse_record08(payload):
    if not isinstance(payload, dict):
        return None, [Diagnostic08("<record>", "expected object")]
    parsed, errors = {}, []
    for field, enum_type in SCHEMA08.items():
        if field not in payload:
            errors.append(Diagnostic08(field, "missing field"))
            continue
        value = payload[field]
        if type(value) is not str:
            errors.append(Diagnostic08(field, "expected string code"))
            continue
        try:
            parsed[field] = enum_type(value)
        except ValueError:
            errors.append(Diagnostic08(field, f"unknown code: {value!r}"))
    extra = set(payload) - set(SCHEMA08)
    for field in sorted(extra):
        errors.append(Diagnostic08(field, "unexpected field"))
    return (None if errors else parsed), errors

### Step 3 — Inspect a valid record and a multiply-invalid record

The parser returns actual members, so later code can compare identities. All problems in a malformed object are reported in a deterministic field order: schema field order followed by sorted unexpected field names.

In [35]:
good08, errors08 = parse_record08({"region": "eu", "tier": "pro", "mode": "safe"})
assert errors08 == []
assert good08["region"] is Region08.EU
assert good08["tier"] is Tier08.PRO

bad08, errors08 = parse_record08({"region": 1, "tier": "vip", "other": 9})
assert bad08 is None
assert [e.field for e in errors08] == ["region", "tier", "mode", "other"]
print(errors08)

[Diagnostic08(field='region', problem='expected string code'), Diagnostic08(field='tier', problem="unknown code: 'vip'"), Diagnostic08(field='mode', problem='missing field'), Diagnostic08(field='other', problem='unexpected field')]


### Step 4 — Test the boundary rather than just the happy path

An unknown member value raises `ValueError`; a wrong-type field is rejected even if its string representation would happen to match. Missing fields and surplus fields are separately reported. The parser never returns a partly usable record.

In [36]:
assert parse_record08([])[1] == [Diagnostic08("<record>", "expected object")]
assert parse_record08({"region": "us", "tier": "free"})[0] is None
assert parse_record08({"region": "us", "tier": "free", "mode": None})[0] is None
assert len(parse_record08({"region": "us", "tier": "free", "mode": "fast", "x": 1})[1]) == 1

### Worked-solution review

**Why it works.** A field-to-enum schema localizes validation and makes error collection systematic. Consumers receive a complete, correctly typed record or no record at all.

**Important boundary / common mistake.** Never return partially validated data as though it were complete. Unknown fields are rejected here by explicit contract; an API that supports extensions might use a different documented policy.

**Cost.** O(k + e log e) for k expected fields and e extra fields, due to sorted diagnostic output.

**Try it yourself:** Add a diagnostic code enum separate from the human-readable diagnostic message; test machine-readable error handling.

In [37]:
assert True
print("Problem 08: all assertions passed")

Problem 08: all assertions passed


## Problem 09 — Implement a streaming CSV importer with row-level diagnostics

**Challenge.** Load a small CSV feed of shipments. Convert the status column to enum members, preserve the row number, and collect malformed records without discarding valid rows.

**Success criteria.** Use `csv.DictReader`, strict headers, row-specific errors, and a clear policy for empty or malformed IDs.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Write a realistic input fixture and domain enum

Real CSV data comes from files; `io.StringIO` lets us test the exact same CSV machinery without needing an external file. The `DictReader` processes rows one at a time, giving us a natural place to attach diagnostics.

In [38]:
import csv
from io import StringIO

class ShipmentStatus09(Enum):
    QUEUED = "queued"
    MOVING = "moving"
    DELIVERED = "delivered"

CSV09 = """shipment_id,status
101,queued
102,moving
bad,delivered
103,unknown
104,delivered
"""
reader09 = csv.DictReader(StringIO(CSV09))
assert reader09.fieldnames == ["shipment_id", "status"]

### Step 2 — Define the output shapes before parsing

Represent a successful row as an immutable record containing its original row number. A failed row becomes a diagnostic, not a half-initialized shipment. Preserve the difference between `KeyError` (a missing field) and `ValueError` (an invalid value).

In [39]:
@dataclass(frozen=True)
class Shipment09:
    line: int
    shipment_id: int
    status: ShipmentStatus09

@dataclass(frozen=True)
class CsvIssue09:
    line: int
    detail: str


def decode_row09(line, row):
    raw_id, raw_status = row["shipment_id"], row["status"]
    if raw_id is None or raw_status is None:
        raise ValueError("missing column value")
    shipment_id = int(raw_id)
    if shipment_id <= 0:
        raise ValueError("shipment_id must be positive")
    return Shipment09(line, shipment_id, ShipmentStatus09(raw_status))

### Step 3 — Build an importer that has a documented failure policy

For this exercise, an invalid header rejects the entire document. Data-level errors are collected per row and do not prevent reading later rows. `reader.line_num` reports the current physical CSV line (useful even when records span multiple physical lines).

In [40]:
def import_shipments09(text):
    reader = csv.DictReader(StringIO(text))
    if reader.fieldnames != ["shipment_id", "status"]:
        raise ValueError("expected headers: shipment_id,status")
    shipments, errors = [], []
    for row in reader:
        line = reader.line_num
        if None in row:  # DictReader stores surplus columns under the None key.
            errors.append(CsvIssue09(line, "extra column(s)"))
            continue
        try:
            shipments.append(decode_row09(line, row))
        except (ValueError, KeyError, TypeError) as exc:
            errors.append(CsvIssue09(line, str(exc)))
    return shipments, errors

### Step 4 — Verify accepted rows, rejected rows, and the header

Only lines with valid IDs and recognized enum values become typed records. The loader continues after a bad row. A wrong header is different from a bad row and is treated as a document-level schema problem.

In [41]:
shipments09, issues09 = import_shipments09(CSV09)
assert [s.shipment_id for s in shipments09] == [101, 102, 104]
assert shipments09[0].status is ShipmentStatus09.QUEUED
assert [e.line for e in issues09] == [4, 5]
assert must_raise(ValueError, lambda: import_shipments09("id,phase\n1,queued\n"))
assert len(import_shipments09("shipment_id,status\n5,queued,extra\n")[1]) == 1
print("Valid:", shipments09, "\nIssues:", issues09)

Valid: [Shipment09(line=2, shipment_id=101, status=<ShipmentStatus09.QUEUED: 'queued'>), Shipment09(line=3, shipment_id=102, status=<ShipmentStatus09.MOVING: 'moving'>), Shipment09(line=6, shipment_id=104, status=<ShipmentStatus09.DELIVERED: 'delivered'>)] 
Issues: [CsvIssue09(line=4, detail="invalid literal for int() with base 10: 'bad'"), CsvIssue09(line=5, detail="'unknown' is not a valid ShipmentStatus09")]


### Step 5 — Ask what “streaming” actually guarantees

`DictReader` consumes its input incrementally. We nevertheless accumulate the successful records and diagnostics in two lists for ease of notebook exploration. For huge feeds, turn this into a generator that yields one `(record, diagnostic)` result at a time.

In [42]:
assert all(isinstance(s.status, ShipmentStatus09) for s in shipments09)
assert set(s.shipment_id for s in shipments09).isdisjoint({103})
print(f"Accepted {len(shipments09)} of {len(shipments09) + len(issues09)} data rows")

Accepted 3 of 5 data rows


### Worked-solution review

**Why it works.** Validate document structure up front, decode one row at a time, and collect row-level issues while retaining the original location. Invalid text never becomes a fake enum member.

**Important boundary / common mistake.** This importer intentionally rejects extra columns and only accepts exact wire values. Real-world systems should document their policy for blank lines, duplicate IDs, encodings, and quoted multiline fields.

**Cost.** O(r) time and O(r) output storage for r rows; the CSV parser itself processes rows incrementally.

**Try it yourself:** Detect duplicate shipment IDs while keeping only the first valid occurrence; include the duplicate line in the diagnostic.

In [43]:
assert True
print("Problem 09: all assertions passed")

Problem 09: all assertions passed


## Problem 10 — Persist enum codes safely in an in-memory SQLite database

**Challenge.** A table stores shipment states as textual codes. Write and read enums without relying on `repr`, implicit database adapters, or a declaration position. Handle legacy codes and reject unknown database values.

**Success criteria.** Create a schema with constraints; serialize explicitly; migrate one documented legacy code; reject unknown and malformed rows on read.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Store a stable external code, not the Python object

SQLite knows standard SQL values, not the semantics of our enum class. The simplest contract is an explicit text column holding `.value`. A `CHECK` constraint can guard some bad writes, but decoding still has to validate database rows.

In [44]:
import sqlite3

class DbState10(Enum):
    NEW = "new"
    SENT = "sent"
    CLOSED = "closed"

connection10 = sqlite3.connect(":memory:")
connection10.execute("""
    CREATE TABLE shipments (
        id INTEGER PRIMARY KEY,
        state TEXT NOT NULL CHECK(state IN ('new','sent','closed','dispatched'))
    )
""")
connection10.commit()
print("Created a constrained in-memory database.")

Created a constrained in-memory database.


### Step 2 — Parameterize statements and encode deliberately

SQL placeholders are for *values*, not table names. Parameterization both clarifies data flow and avoids interpolating user-provided strings into SQL. Validate the Python enum type before calling the database.

In [45]:
def write_state10(conn, row_id, state: DbState10):
    if not isinstance(state, DbState10):
        raise TypeError("state must be DbState10")
    conn.execute(
        "INSERT INTO shipments(id, state) VALUES (?, ?)",
        (row_id, state.value),
    )
    conn.commit()

write_state10(connection10, 1, DbState10.NEW)
write_state10(connection10, 2, DbState10.SENT)
assert connection10.execute("SELECT state FROM shipments WHERE id=1").fetchone() == ("new",)

### Step 3 — Give a legacy value an explicit migration policy

Suppose an older release stored `dispatched`, and the documented migration maps it to `SENT`. This translation belongs at a data boundary, not in unrelated business logic. The mapping lists **only** the aliases we have deliberately approved.

In [46]:
LEGACY10 = {"dispatched": "sent"}

def decode_db_state10(raw):
    if type(raw) is not str:
        raise TypeError("database state must be text")
    canonical = LEGACY10.get(raw, raw)
    return DbState10(canonical)

connection10.execute("INSERT INTO shipments(id,state) VALUES (?,?)", (3, "dispatched"))
connection10.commit()
assert decode_db_state10("dispatched") is DbState10.SENT
assert must_raise(ValueError, lambda: decode_db_state10("vanished"))

### Step 4 — Build a strict reader and exercise both directions

When a row is absent, raise `LookupError`; for a malformed stored value, enum construction raises `ValueError`. Keep those failure meanings separate so the caller can distinguish “not found” from “bad persisted data.”

In [47]:
def read_state10(conn, row_id):
    result = conn.execute(
        "SELECT state FROM shipments WHERE id = ?", (row_id,)
    ).fetchone()
    if result is None:
        raise LookupError(f"shipment {row_id} not found")
    return decode_db_state10(result[0])

assert read_state10(connection10, 1) is DbState10.NEW
assert read_state10(connection10, 3) is DbState10.SENT
assert must_raise(LookupError, lambda: read_state10(connection10, 999))
assert must_raise(TypeError, lambda: write_state10(connection10, 9, "new"))

### Step 5 — Check that the SQL constraint is not your only defense

An invalid insert is blocked by the current schema. However, database files may predate a schema, arrive from an external source, or be imported with different constraints. The explicit decoder remains necessary even when SQL also validates writes.

In [48]:
must_raise(
    sqlite3.IntegrityError,
    lambda: connection10.execute("INSERT INTO shipments(id,state) VALUES (?,?)", (7, "invalid"))
)
connection10.rollback()
assert connection10.execute("SELECT COUNT(*) FROM shipments").fetchone()[0] == 3
connection10.close()
print("Database closed after assertions.")

Database closed after assertions.


### Worked-solution review

**Why it works.** Persist a deliberate text representation, use parameterized statements, make migrations explicit, and decode through the enum class. SQL constraints and Python validation complement each other.

**Important boundary / common mistake.** Avoid persisting `str(member)` or the automatic integer position. Do not use an open-ended, case-insensitive fallback for database migrations unless your documented data contract calls for one.

**Try it yourself:** Add a migration transaction that permanently rewrites legacy `dispatched` rows to `sent`, then remove the legacy code from the SQL constraint.

In [49]:
assert True
print("Problem 10: all assertions passed")

Problem 10: all assertions passed


## Problem 11 — Design and validate a compact binary message protocol

**Challenge.** A device sends a four-byte frame: one version byte, one opcode byte, and a big-endian two-byte payload length. Parse it into a typed enum opcode, reject unknown versions and codes, and detect length mismatches.

**Success criteria.** Implement `struct.pack` / `struct.unpack`, strict bounds checking, a typed frame record, and byte-for-byte round trips.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Specify bytes before implementing code

A binary format needs a precise contract. We choose `!BBH`: network byte order, unsigned byte version, unsigned byte opcode, and unsigned short length. The header is four bytes; payload follows immediately. Unknown opcodes are never silently coerced.

In [50]:
import struct

class Opcode11(Enum):
    PING = 1
    DATA = 2
    ACK = 3

HEADER11 = struct.Struct("!BBH")
VERSION11 = 1
assert HEADER11.size == 4
assert struct.pack("!H", 258) == b"\x01\x02"

### Step 2 — Represent a decoded frame with an enum member

Keep raw bytes as `bytes`; make the opcode an actual `Opcode11` member. Freezing the dataclass prevents accidental field reassignment, which is useful for an immutable message object.

In [51]:
@dataclass(frozen=True)
class Frame11:
    opcode: Opcode11
    payload: bytes


def encode_frame11(frame: Frame11) -> bytes:
    if not isinstance(frame, Frame11) or not isinstance(frame.opcode, Opcode11):
        raise TypeError("expected a typed Frame11")
    if type(frame.payload) is not bytes:
        raise TypeError("payload must be bytes")
    if len(frame.payload) > 65535:
        raise ValueError("payload is too long")
    return HEADER11.pack(VERSION11, frame.opcode.value, len(frame.payload)) + frame.payload

wire11 = encode_frame11(Frame11(Opcode11.DATA, b"abc"))
assert wire11 == b"\x01\x02\x00\x03abc"

### Step 3 — Parse header length and validate in the right order

Before unpacking, check that a full header exists. Verify the version, then convert the opcode through enum value lookup. Finally check that the declared payload length *equals* the remaining data length; both truncated and surplus bytes are invalid under this protocol.

In [52]:
def decode_frame11(data: bytes) -> Frame11:
    if type(data) is not bytes:
        raise TypeError("binary frame required")
    if len(data) < HEADER11.size:
        raise ValueError("truncated header")
    version, raw_code, expected_length = HEADER11.unpack(data[:HEADER11.size])
    if version != VERSION11:
        raise ValueError(f"unsupported version {version}")
    opcode = Opcode11(raw_code)
    payload = data[HEADER11.size:]
    if len(payload) != expected_length:
        raise ValueError("payload length mismatch")
    return Frame11(opcode, payload)

assert decode_frame11(wire11) == Frame11(Opcode11.DATA, b"abc")

### Step 4 — Probe corrupt packets without crashing the notebook

The decoder intentionally raises exceptions. Tests catch only the expected type, confirming that a malformed wire packet is rejected rather than converted into a plausible application command.

In [53]:
assert must_raise(ValueError, lambda: decode_frame11(b"\x01"))
assert must_raise(ValueError, lambda: decode_frame11(b"\x02\x01\x00\x00"))
assert must_raise(ValueError, lambda: decode_frame11(b"\x01\xff\x00\x00"))
assert must_raise(ValueError, lambda: decode_frame11(b"\x01\x02\x00\x04abc"))
assert must_raise(ValueError, lambda: decode_frame11(wire11 + b"!"))
assert must_raise(TypeError, lambda: decode_frame11("not bytes"))

### Step 5 — Check the round-trip property on every opcode

A simple exhaustive test over the enum catches mismatches between implementation and declared opcode values. Include empty and binary payloads; do not assume payloads are UTF-8 text.

In [54]:
for opcode11 in Opcode11:
    for payload11 in (b"", b"\x00\xff", b"hello"):
        frame11 = Frame11(opcode11, payload11)
        assert decode_frame11(encode_frame11(frame11)) == frame11
print("All opcode/payload round trips passed.")

All opcode/payload round trips passed.


### Worked-solution review

**Why it works.** A format specification, exact byte sizes, stable numeric enum codes, and explicit validation prevent truncated or unknown data from being interpreted as valid commands.

**Important boundary / common mistake.** A production network parser may require checksum/authentication, limits below 65,535 bytes, framing across multiple reads, and version negotiation. This exercise verifies one complete frame, not a stream of partial reads.

**Cost.** Encoding and decoding each take O(p) time and storage for a payload of p bytes.

**Try it yourself:** Add a checksum byte and verify it before constructing a `Frame11` object.

In [55]:
assert True
print("Problem 11: all assertions passed")

Problem 11: all assertions passed


## Problem 12 — Write a safe arithmetic parser with enum token kinds

**Challenge.** Evaluate expressions containing nonnegative integers, `+`, `*`, and parentheses. Represent tokens with an enum, respect multiplication precedence, and reject unrecognized characters without using `eval`.

**Success criteria.** Tokenize input, implement a small recursive-descent grammar, evaluate valid expressions, and reject invalid syntax and division-like injection.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Specify the grammar before writing a parser

We will use `expression := term (PLUS term)*`, `term := factor (STAR factor)*`, and `factor := INTEGER | LPAREN expression RPAREN`. Separating levels encodes operator precedence directly: `*` binds more tightly than `+`.

In [56]:
class TokenKind12(Enum):
    INTEGER = "integer"
    PLUS = "+"
    STAR = "*"
    LPAREN = "("
    RPAREN = ")"
    EOF = "<eof>"

@dataclass(frozen=True)
class Token12:
    kind: TokenKind12
    value: int | None = None
    position: int = 0

assert Token12(TokenKind12.INTEGER, 12).kind is TokenKind12.INTEGER

### Step 2 — Convert a character stream into typed tokens

Tokenization is a boundary: only digits, spaces, and exactly four punctuation characters are supported. The tokenizer appends one EOF token so the parser can check for trailing junk after a valid expression.

In [57]:
PUNCTUATION12 = {
    "+": TokenKind12.PLUS,
    "*": TokenKind12.STAR,
    "(": TokenKind12.LPAREN,
    ")": TokenKind12.RPAREN,
}

def tokenize12(text: str):
    if type(text) is not str:
        raise TypeError("expression must be a string")
    tokens, index = [], 0
    while index < len(text):
        char = text[index]
        if char.isspace():
            index += 1
        elif "0" <= char <= "9":
            start = index
            while index < len(text) and "0" <= text[index] <= "9":
                index += 1
            tokens.append(Token12(TokenKind12.INTEGER, int(text[start:index]), start))
        elif char in PUNCTUATION12:
            tokens.append(Token12(PUNCTUATION12[char], position=index))
            index += 1
        else:
            raise ValueError(f"unrecognized character at {index}: {char!r}")
    tokens.append(Token12(TokenKind12.EOF, position=len(text)))
    return tokens

assert [t.kind for t in tokenize12("2+3*")] == [
    TokenKind12.INTEGER, TokenKind12.PLUS, TokenKind12.INTEGER,
    TokenKind12.STAR, TokenKind12.EOF,
]

### Step 3 — Parse factors first, since they have the tightest structure

A parser cursor points at one token. `factor` consumes an integer or a parenthesized expression. If anything else appears where a factor is expected, the parser raises `ValueError` with a useful position.

In [58]:
class ArithmeticParser12:
    def __init__(self, tokens):
        self.tokens = tokens
        self.index = 0

    def current(self):
        return self.tokens[self.index]

    def accept(self, kind):
        if self.current().kind is kind:
            self.index += 1
            return True
        return False

    def factor(self):
        token = self.current()
        if self.accept(TokenKind12.INTEGER):
            return token.value
        if self.accept(TokenKind12.LPAREN):
            value = self.expression()
            if not self.accept(TokenKind12.RPAREN):
                raise ValueError(f"expected ')' at {self.current().position}")
            return value
        raise ValueError(f"expected number or '(' at {token.position}")

### Step 4 — Implement precedence with two small loops

`term` consumes multiplication chains, and `expression` consumes addition chains. Both are left-associative. Because `expression` calls `term`, the grammar handles multiplication before the surrounding additions.

In [59]:
class CompleteArithmeticParser12(ArithmeticParser12):
    def term(self):
        result = self.factor()
        while self.accept(TokenKind12.STAR):
            result *= self.factor()
        return result

    def expression(self):
        result = self.term()
        while self.accept(TokenKind12.PLUS):
            result += self.term()
        return result


def evaluate12(text):
    parser = CompleteArithmeticParser12(tokenize12(text))
    result = parser.expression()
    if parser.current().kind is not TokenKind12.EOF:
        raise ValueError(f"trailing token at {parser.current().position}")
    return result

assert evaluate12("2+3*4") == 14
assert evaluate12("(2+3)*4") == 20

### Step 5 — Verify syntax errors and injection attempts

The grammar does not recognize names, attribute access, minus signs, or function calls. The tokenization step rejects unknown characters *before* arithmetic evaluation. Importantly, no Python source is evaluated as executable code.

In [60]:
valid12 = {"0": 0, "42": 42, "1+2+3": 6, "2*(3+4)": 14, " 9 * 2 + 1 ": 19}
for expression12_text, result12 in valid12.items():
    assert evaluate12(expression12_text) == result12
for broken12 in ("", "2+", "*3", "(2+3", "2+3)", "2/3", "__import__('os')", "2 3"):
    must_raise(ValueError, lambda broken12=broken12: evaluate12(broken12))
print("Arithmetic grammar checks passed.")

Arithmetic grammar checks passed.


### Worked-solution review

**Why it works.** Enum token kinds keep the parser vocabulary precise. The three grammar levels implement precedence directly; a final EOF check prevents silently accepting extra tokens.

**Important boundary / common mistake.** This intentionally tiny language does not support unary operators, subtraction, decimals, or arbitrarily large input safety limits. A production service should also impose an input-size and nesting-depth budget.

**Cost.** O(n) time and O(n) token storage for input length n, with recursion depth proportional to parenthesis nesting.

**Try it yourself:** Add subtraction and unary minus as *distinct grammar rules* and test `2*-3` and `-(2+3)`; do not pass the original text to `eval`.

In [61]:
assert True
print("Problem 12: all assertions passed")

Problem 12: all assertions passed


## Problem 13 — Coordinate asynchronous work using typed lifecycle events

**Challenge.** Several jobs run concurrently with `asyncio`. Give each job a typed lifecycle, capture success or failure, and demonstrate that completed task results can be collected without depending on finish order.

**Success criteria.** Model lifecycle with enums; use `asyncio.gather`; avoid external services and sleeps; guarantee the terminal state is consistent with the result.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Describe states without coupling them to scheduling order

The enum is a vocabulary: `CREATED`, `RUNNING`, `SUCCEEDED`, and `FAILED`. Concurrency timing is controlled by the event loop, not by the order in which enum members are declared. We will use `await asyncio.sleep(0)` solely to yield scheduling control deterministically in a local notebook.

In [62]:
import asyncio

class WorkState13(Enum):
    CREATED = "created"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"

@dataclass(frozen=True)
class WorkResult13:
    job_id: int
    state: WorkState13
    value: int | None
    error: str | None

assert WorkState13.SUCCEEDED is not WorkState13.FAILED

### Step 2 — Represent an individual asynchronous computation

A coroutine returns a typed result rather than mutating a global status dictionary. Catch only the expected `ValueError` from negative inputs. Unexpected programming bugs must not be masked as ordinary data errors.

In [63]:
async def execute_job13(job_id: int, amount: int) -> WorkResult13:
    state = WorkState13.CREATED
    state = WorkState13.RUNNING
    await asyncio.sleep(0)
    try:
        if amount < 0:
            raise ValueError("amount must not be negative")
        answer = amount * amount
    except ValueError as exc:
        return WorkResult13(job_id, WorkState13.FAILED, None, str(exc))
    return WorkResult13(job_id, WorkState13.SUCCEEDED, answer, None)

assert asyncio.iscoroutinefunction(execute_job13)

### Step 3 — Gather tasks in a way that works inside Jupyter

Jupyter notebooks normally have a running event loop, so calling `asyncio.run()` in a code cell often fails. A notebook cell supports top-level `await`. `gather` returns results in the order of input awaitables, not in an assumed wall-clock completion order.

In [64]:
jobs13 = [(1, 3), (2, -1), (3, 5)]
results13 = await asyncio.gather(
    *(execute_job13(job_id, amount) for job_id, amount in jobs13)
)
assert [result.job_id for result in results13] == [1, 2, 3]
assert [result.state for result in results13] == [
    WorkState13.SUCCEEDED, WorkState13.FAILED, WorkState13.SUCCEEDED
]
print(results13)

[WorkResult13(job_id=1, state=<WorkState13.SUCCEEDED: 'succeeded'>, value=9, error=None), WorkResult13(job_id=2, state=<WorkState13.FAILED: 'failed'>, value=None, error='amount must not be negative'), WorkResult13(job_id=3, state=<WorkState13.SUCCEEDED: 'succeeded'>, value=25, error=None)]


### Step 4 — Assert an invariant over *all* completed jobs

The invariant is more important than any particular execution order: success has a numerical answer and no error; failure has no answer and a diagnostic. Express it in one reusable predicate and check the entire batch.

In [65]:
def valid_result13(result):
    if result.state is WorkState13.SUCCEEDED:
        return result.value is not None and result.error is None
    if result.state is WorkState13.FAILED:
        return result.value is None and isinstance(result.error, str)
    return False

assert all(valid_result13(result) for result in results13)
assert [x.value for x in results13 if x.state is WorkState13.SUCCEEDED] == [9, 25]

### Worked-solution review

**Why it works.** Enums provide clear lifecycle labels; immutable result objects prevent a caller from observing a partially updated global state. Top-level `await` is the notebook-friendly concurrency API.

**Important boundary / common mistake.** The example returns only terminal results, not a live state machine. Production work queues additionally need cancellation, timeouts, retries, and idempotency; adding those requires an explicit contract for each transition.

**Try it yourself:** Add a fifth state `CANCELLED` to a new version and extend the result invariant without treating cancellation as a success.

In [66]:
assert True
print("Problem 13: all assertions passed")

Problem 13: all assertions passed


## Problem 14 — Replay an append-only inventory ledger using enum event kinds

**Challenge.** An inventory system stores events instead of overwriting quantity. Rebuild the current quantity from events, reject impossible operations, and detect events arriving out of order.

**Success criteria.** Use typed events, contiguous sequence numbers, a pure replay function, and negative tests for underflow and invalid ordering.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Separate an event from current state

An event is a historical fact: receive, ship, or correct. Each event has a sequence number and a nonnegative quantity. The ledger is append-only; the current balance is a *derived* result. An enum makes event kinds explicit.

In [67]:
class InventoryEvent14(Enum):
    RECEIVE = "receive"
    SHIP = "ship"
    ADJUST = "adjust"

@dataclass(frozen=True)
class LedgerEntry14:
    sequence: int
    kind: InventoryEvent14
    quantity: int

ledger14 = [
    LedgerEntry14(1, InventoryEvent14.RECEIVE, 10),
    LedgerEntry14(2, InventoryEvent14.SHIP, 4),
    LedgerEntry14(3, InventoryEvent14.ADJUST, 2),
]
assert all(isinstance(x.kind, InventoryEvent14) for x in ledger14)

### Step 2 — Write a pure replay operation

Here `ADJUST` means **set the absolute stock level**, not add a delta. That meaning is part of the event contract. Require consecutive positive sequence numbers, valid member types, and nonnegative exact integers to avoid ambiguous ledger interpretation.

In [68]:
def replay_ledger14(events):
    balance, expected_sequence = 0, 1
    for event in events:
        if not isinstance(event, LedgerEntry14):
            raise TypeError("ledger contains a non-entry")
        if type(event.sequence) is not int or event.sequence != expected_sequence:
            raise ValueError("sequence numbers must be contiguous starting at one")
        if not isinstance(event.kind, InventoryEvent14):
            raise TypeError("unknown event kind")
        if type(event.quantity) is not int or event.quantity < 0:
            raise ValueError("quantity must be a nonnegative integer")
        if event.kind is InventoryEvent14.RECEIVE:
            balance += event.quantity
        elif event.kind is InventoryEvent14.SHIP:
            if event.quantity > balance:
                raise ValueError("cannot ship more than available stock")
            balance -= event.quantity
        elif event.kind is InventoryEvent14.ADJUST:
            balance = event.quantity
        expected_sequence += 1
    return balance

assert replay_ledger14(ledger14) == 2

### Step 3 — Test history integrity and impossible inventory

Negative tests are part of the solution: they show a shipped quantity cannot put stock below zero and a sequence gap cannot be ignored. `bool` is excluded even though `isinstance(True, int)` is true in Python.

In [69]:
must_raise(ValueError, lambda: replay_ledger14([
    LedgerEntry14(1, InventoryEvent14.SHIP, 1)
]))
must_raise(ValueError, lambda: replay_ledger14([
    LedgerEntry14(2, InventoryEvent14.RECEIVE, 1)
]))
must_raise(ValueError, lambda: replay_ledger14([
    LedgerEntry14(1, InventoryEvent14.RECEIVE, True)
]))
assert replay_ledger14([]) == 0

### Step 4 — Prove that replay depends on the ordered history

The same kinds and quantities in a different order are *not* necessarily equivalent. An initial shipment with no stock is rejected even if a later receipt would have made the final sum nonnegative.

In [70]:
ordered14 = [
    LedgerEntry14(1, InventoryEvent14.RECEIVE, 5),
    LedgerEntry14(2, InventoryEvent14.SHIP, 4),
]
reordered14 = [
    LedgerEntry14(1, InventoryEvent14.SHIP, 4),
    LedgerEntry14(2, InventoryEvent14.RECEIVE, 5),
]
assert replay_ledger14(ordered14) == 1
must_raise(ValueError, lambda: replay_ledger14(reordered14))

ValueError('cannot ship more than available stock')

### Worked-solution review

**Why it works.** Event enums make history categories explicit. A deterministic, side-effect-free replay function reconstructs state and validates every prefix of the event sequence, not merely the final arithmetic total.

**Important boundary / common mistake.** Changing the meaning of an event such as `ADJUST` would corrupt historical interpretation. Event schemas need versioning and migrations, just like externally persisted enum values.

**Cost.** O(n) time and O(1) auxiliary space for n events, excluding storage of the input list.

**Try it yourself:** Add a `RETURN` event whose quantity increases stock, and test that old histories still replay identically.

In [71]:
assert True
print("Problem 14: all assertions passed")

Problem 14: all assertions passed


## Problem 15 — Exhaustively verify a cross-product of two enum domains

**Challenge.** A transport layer has three modes and three data formats. Document and verify which nine combinations are supported without hiding missing cases in default behavior.

**Success criteria.** Enumerate the full Cartesian product, define a compatibility policy, and fail if a new enum member has not been considered.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Define orthogonal concepts separately

One enum represents *transport* and the other represents *format*. Avoid making nine composite member names: that would conflate two independent decisions and make extensions awkward. `itertools.product` generates every combination explicitly.

In [72]:
from itertools import product

class Transport15(Enum):
    HTTP = "http"
    MQTT = "mqtt"
    FILE = "file"

class Format15(Enum):
    JSON = "json"
    BINARY = "binary"
    CSV = "csv"

all_pairs15 = list(product(Transport15, Format15))
assert len(all_pairs15) == 9
print([(a.name, b.name) for a, b in all_pairs15])

[('HTTP', 'JSON'), ('HTTP', 'BINARY'), ('HTTP', 'CSV'), ('MQTT', 'JSON'), ('MQTT', 'BINARY'), ('MQTT', 'CSV'), ('FILE', 'JSON'), ('FILE', 'BINARY'), ('FILE', 'CSV')]


### Step 2 — Represent documented compatibility as data

Suppose HTTP supports JSON and binary, MQTT supports only binary, and file storage supports JSON and CSV. The set expresses the published policy without relying on ordinal enum values or comparing members from different types.

In [73]:
SUPPORTED15 = {
    (Transport15.HTTP, Format15.JSON),
    (Transport15.HTTP, Format15.BINARY),
    (Transport15.MQTT, Format15.BINARY),
    (Transport15.FILE, Format15.JSON),
    (Transport15.FILE, Format15.CSV),
}


def supports15(transport, data_format):
    if not isinstance(transport, Transport15):
        raise TypeError("transport must be Transport15")
    if not isinstance(data_format, Format15):
        raise TypeError("format must be Format15")
    return (transport, data_format) in SUPPORTED15

assert supports15(Transport15.MQTT, Format15.BINARY)
assert not supports15(Transport15.MQTT, Format15.CSV)

### Step 3 — Make unsupported cases visible

A negative result is a deliberate policy, not an omitted branch. Build a truth table and assert the exact number of supported and unsupported combinations. This is a small model-based test of the complete input space.

In [74]:
truth_table15 = [
    (transport, data_format, supports15(transport, data_format))
    for transport, data_format in all_pairs15
]
assert sum(allowed for _, _, allowed in truth_table15) == 5
assert sum(not allowed for _, _, allowed in truth_table15) == 4
for transport, data_format, allowed in truth_table15:
    print(f"{transport.name:5s} / {data_format.name:6s} -> {allowed}")

HTTP  / JSON   -> True
HTTP  / BINARY -> True
HTTP  / CSV    -> False
MQTT  / JSON   -> False
MQTT  / BINARY -> True
MQTT  / CSV    -> False
FILE  / JSON   -> True
FILE  / BINARY -> False
FILE  / CSV    -> True


### Step 4 — Catch an incomplete policy after an enum grows

If a new member is introduced, an old policy set does not automatically cover it. An explicit pair-level decision table can be tested against the Cartesian product. Here we construct a complete table by assigning `False` to unsupported pairs and check its keys.

In [75]:
DECISIONS15 = {
    pair: pair in SUPPORTED15 for pair in all_pairs15
}
assert set(DECISIONS15) == set(product(Transport15, Format15))
assert all(DECISIONS15[pair] == supports15(*pair) for pair in DECISIONS15)

def ensure_complete15(enum_a, enum_b, decisions):
    if set(decisions) != set(product(enum_a, enum_b)):
        raise ValueError("compatibility matrix is incomplete")

ensure_complete15(Transport15, Format15, DECISIONS15)
missing15 = dict(DECISIONS15)
missing15.pop((Transport15.HTTP, Format15.JSON))
must_raise(ValueError, lambda: ensure_complete15(Transport15, Format15, missing15))

ValueError('compatibility matrix is incomplete')

### Worked-solution review

**Why it works.** Use separate enums for separate dimensions, enumerate the cross-product, and require an explicit policy decision for every pair. The coverage check exposes incomplete updates when the allowed vocabulary grows.

**Important boundary / common mistake.** A default `return False` can hide unreviewed combinations. In safety-sensitive or contract-sensitive logic, distinguish an intentionally unsupported pair from a pair that nobody evaluated.

**Cost.** The exhaustive check needs O(a × b) time and storage for enums with a and b members.

**Try it yourself:** Add an explicit reason string to each unsupported pair and require every denial to have an explanation.

In [76]:
assert True
print("Problem 15: all assertions passed")

Problem 15: all assertions passed


## Problem 16 — Capstone: a multi-format telemetry gateway

**Challenge.** Combine the lessons into a gateway that accepts JSON text or a compact binary frame, converts both to one typed event model, validates severity and measurement bounds, and counts accepted events without losing diagnostics.

**Success criteria.** Define an enum-based contract, implement two independent decoders, normalize into one frozen dataclass, process a batch containing both formats, and prove equivalent payloads produce equivalent typed events.

**How this tutorial works:** Read the reasoning before each small implementation cell. Predict the outcome, run the cell, and then inspect the assertions. The final review explains why the implementation works and what could go wrong.

### Step 1 — Write the integration contract *before* either decoder

The gateway recognizes measurements `temperature` and `pressure`; severity is `info` or `alert`. A JSON payload carries string codes. A binary packet uses one-byte numeric codes plus a signed two-byte measurement. Both representations must produce **the same internal object**. We will not use any network connection or external file.

In [77]:
import json

class Metric16(Enum):
    TEMPERATURE = "temperature"
    PRESSURE = "pressure"

class Severity16(Enum):
    INFO = "info"
    ALERT = "alert"

METRIC_CODE16 = {Metric16.TEMPERATURE: 1, Metric16.PRESSURE: 2}
SEVERITY_CODE16 = {Severity16.INFO: 1, Severity16.ALERT: 2}
METRIC_BY_CODE16 = {code: metric for metric, code in METRIC_CODE16.items()}
SEVERITY_BY_CODE16 = {code: severity for severity, code in SEVERITY_CODE16.items()}

assert len(METRIC_BY_CODE16) == len(Metric16)
assert len(SEVERITY_BY_CODE16) == len(Severity16)

### Step 2 — Define the canonical typed event and its invariants

The two decoders will share one validation function. The numeric value is an **exact integer in the inclusive range −1000 to 1000**; this rejects booleans and floats rather than silently truncating them. Consistent invariants are essential if representations are to be interchangeable.

In [78]:
@dataclass(frozen=True)
class Telemetry16:
    metric: Metric16
    severity: Severity16
    reading: int


def make_telemetry16(metric, severity, reading):
    if not isinstance(metric, Metric16):
        raise TypeError("metric must be Metric16")
    if not isinstance(severity, Severity16):
        raise TypeError("severity must be Severity16")
    if type(reading) is not int or not -1000 <= reading <= 1000:
        raise ValueError("reading must be an integer between -1000 and 1000")
    return Telemetry16(metric, severity, reading)

assert make_telemetry16(Metric16.PRESSURE, Severity16.INFO, 7).reading == 7
must_raise(ValueError, lambda: make_telemetry16(Metric16.PRESSURE, Severity16.INFO, True))

ValueError('reading must be an integer between -1000 and 1000')

### Step 3 — Implement the JSON decoder with exact schema checks

JSON is untrusted input: validate the *shape*, required field names, string types, and integer range. Enum lookups then reject unfamiliar metric/severity values. Rejecting unknown fields here is a deliberate documented policy, not an intrinsic enum behavior.

In [79]:
JSON_FIELDS16 = {"metric", "severity", "reading"}

def parse_json16(text: str) -> Telemetry16:
    if type(text) is not str:
        raise TypeError("JSON payload must be text")
    payload = json.loads(text)
    if not isinstance(payload, dict) or set(payload) != JSON_FIELDS16:
        raise ValueError("wrong telemetry object fields")
    if type(payload["metric"]) is not str or type(payload["severity"]) is not str:
        raise TypeError("enum wire codes must be strings")
    return make_telemetry16(
        Metric16(payload["metric"]),
        Severity16(payload["severity"]),
        payload["reading"],
    )

json_payload16 = '{"metric":"temperature","severity":"alert","reading":25}'
assert parse_json16(json_payload16) == Telemetry16(Metric16.TEMPERATURE, Severity16.ALERT, 25)

### Step 4 — Implement the binary decoder without assuming text rules

Binary format: exactly four bytes `!BBh`, where the first two unsigned bytes are metric and severity codes, followed by one signed 16-bit reading. Validate the packet length *before* unpacking, then map codes explicitly into their enum members.

In [80]:
PACKET16 = struct.Struct("!BBh")
assert PACKET16.size == 4

def parse_binary16(packet: bytes) -> Telemetry16:
    if type(packet) is not bytes:
        raise TypeError("binary payload must be bytes")
    if len(packet) != PACKET16.size:
        raise ValueError("binary telemetry packet must be exactly four bytes")
    metric_code, severity_code, reading = PACKET16.unpack(packet)
    try:
        metric = METRIC_BY_CODE16[metric_code]
        severity = SEVERITY_BY_CODE16[severity_code]
    except KeyError as exc:
        raise ValueError(f"unknown telemetry code: {exc.args[0]}") from exc
    return make_telemetry16(metric, severity, reading)

binary_payload16 = PACKET16.pack(1, 2, 25)
assert parse_binary16(binary_payload16) == parse_json16(json_payload16)

### Step 5 — Provide explicit serializers and test round trips

A real boundary needs both directions. JSON serializes `.value` fields; binary serializes published numeric codes. `sort_keys=True` makes demonstration output reproducible; parsing does not depend on JSON object key order.

In [81]:
def encode_json16(event: Telemetry16) -> str:
    event = make_telemetry16(event.metric, event.severity, event.reading)
    return json.dumps({
        "metric": event.metric.value,
        "severity": event.severity.value,
        "reading": event.reading,
    }, sort_keys=True)


def encode_binary16(event: Telemetry16) -> bytes:
    event = make_telemetry16(event.metric, event.severity, event.reading)
    return PACKET16.pack(
        METRIC_CODE16[event.metric],
        SEVERITY_CODE16[event.severity],
        event.reading,
    )

for metric16 in Metric16:
    for severity16 in Severity16:
        for reading16 in (-1000, 0, 1000):
            event16 = Telemetry16(metric16, severity16, reading16)
            assert parse_json16(encode_json16(event16)) == event16
            assert parse_binary16(encode_binary16(event16)) == event16
assert parse_json16(encode_json16(parse_binary16(binary_payload16))) == parse_binary16(binary_payload16)

### Step 6 — Process mixed inputs while retaining useful failure information

The batch format is a list of `(format, payload)` pairs; each format must be an explicit enum member. A malformed item creates a diagnostic containing its index. Catch known input-level errors, not all exceptions, so programming mistakes still surface during development.

In [82]:
class InputFormat16(Enum):
    JSON = "json"
    BINARY = "binary"

@dataclass(frozen=True)
class GatewayIssue16:
    index: int
    detail: str


def process_batch16(items):
    accepted, issues = [], []
    for index, item in enumerate(items):
        try:
            kind, payload = item
            if kind is InputFormat16.JSON:
                accepted.append(parse_json16(payload))
            elif kind is InputFormat16.BINARY:
                accepted.append(parse_binary16(payload))
            else:
                raise ValueError("unsupported format member")
        except (TypeError, ValueError, KeyError, json.JSONDecodeError) as exc:
            issues.append(GatewayIssue16(index, str(exc)))
    return accepted, issues

batch16 = [
    (InputFormat16.JSON, json_payload16),
    (InputFormat16.BINARY, binary_payload16),
    (InputFormat16.JSON, '{"metric":"unknown","severity":"info","reading":4}'),
    (InputFormat16.BINARY, b"short"),
]
accepted16, issues16 = process_batch16(batch16)
assert len(accepted16) == 2
assert accepted16[0] == accepted16[1]
assert [issue.index for issue in issues16] == [2, 3]
print("Accepted:", accepted16, "\nDiagnostics:", issues16)

Accepted: [Telemetry16(metric=<Metric16.TEMPERATURE: 'temperature'>, severity=<Severity16.ALERT: 'alert'>, reading=25), Telemetry16(metric=<Metric16.TEMPERATURE: 'temperature'>, severity=<Severity16.ALERT: 'alert'>, reading=25)] 
Diagnostics: [GatewayIssue16(index=2, detail="'unknown' is not a valid Metric16"), GatewayIssue16(index=3, detail='binary telemetry packet must be exactly four bytes')]


### Step 7 — Summarize accepted data using enum members as dictionary keys

After decoding, downstream code operates exclusively on typed events. There is no need to remember which transport produced a measurement. Counting by the `Metric16` enum itself avoids fragile string spelling checks in business logic.

In [83]:
counts16 = Counter(event.metric for event in accepted16)
alerts16 = [event for event in accepted16 if event.severity is Severity16.ALERT]
assert counts16[Metric16.TEMPERATURE] == 2
assert counts16[Metric16.PRESSURE] == 0
assert len(alerts16) == 2
assert set(counts16).issubset(set(Metric16))

### Step 8 — Run a compact security and corruption regression suite

Exercise schema pollution, wrong types, booleans, unknown numeric codes, and extra binary bytes. A boundary should fail predictably, not insert a plausible-looking member or truncate data.

In [84]:
must_raise(ValueError, lambda: parse_json16('{"metric":"temperature","severity":"info","reading":1,"extra":7}'))
must_raise(ValueError, lambda: parse_json16('{"metric":"temperature","severity":"info","reading":true}'))
must_raise(TypeError, lambda: parse_json16('{"metric":1,"severity":"info","reading":1}'))
must_raise(ValueError, lambda: parse_binary16(PACKET16.pack(99, 1, 4)))
must_raise(ValueError, lambda: parse_binary16(PACKET16.pack(1, 2, 1001)))
must_raise(ValueError, lambda: parse_binary16(binary_payload16 + b"!"))
assert len(accepted16) + len(issues16) == len(batch16)
print("Capstone contract and corruption checks passed.")

Capstone contract and corruption checks passed.


### Worked-solution review

**Why it works.** One canonical dataclass plus a shared validation function makes both JSON and binary representations converge on the same meaning. Explicit serializers, decoders, and diagnostics isolate external formats from the application model.

**Important boundary / common mistake.** The example is not a production network receiver: it omits authentication, replay protection, packet framing, input-size limits, and a persistence strategy. Validate those separately before deploying a real gateway.

**Cost.** For n fixed-size binary packets, batch work is O(n); JSON decoding additionally depends on each input text length.

### Capstone extension: a genuine design exercise

Without changing the two existing serializers, introduce `Metric16.HUMIDITY` in a new version and answer these questions **in prose before code**:

1. Which explicit numeric code does the new metric receive? How is that code documented?
2. What should an older decoder do with the new code: reject it, preserve an unknown payload, or negotiate a version? Pick a contract and justify it.
3. Which loops above will automatically test the new member, and which mappings might become incomplete?
4. How will you test migrations for persisted events created by the older release?

This is intentionally open-ended: the correct design depends on your compatibility requirements. The existing notebook deliberately treats unknown codes as errors.

In [85]:
assert True
print("Problem 16: all assertions passed")

Problem 16: all assertions passed


---

## Solution checklist: what you should now be able to do

- Distinguish **identity**, **name**, **value**, and the defining enum **class** even when two class declarations look identical.
- Create enums dynamically from validated trusted data without losing duplicate-name information.
- Accept human-friendly labels without accidental normalization collisions.
- Share methods through safe mixins while keeping enums type-distinct.
- Generate values intentionally and recognize the schema-compatibility hazards of generated wire codes.
- Explain how aliases interact with equality, `__members__`, iteration, and hashability.
- Match qualified enum members safely, including exhaustive dispatch checks.
- Produce diagnostics rather than partially validated records for bad API and CSV data.
- Persist stable values in SQLite and handle explicitly documented legacy codes.
- Validate binary framing, parse a small expression grammar, and model asynchronous results.
- Replay typed ledger events, exhaustively test compatibility matrices, and build a multi-format gateway.

**One-page mental model:** names and values are external-facing representations; members are typed singleton-like domain objects. Choose the representation at a boundary, validate it there, and keep the rest of your program operating on real enum members.

## Additional practice — five independent prompts (without new solution code)

These are optional exercises to attempt after finishing the 16 fully worked problems.

**A. Duplicate manifest entries.** Extend Problem 2 so that the schema includes human-readable descriptions. Reject duplicate codes even when the descriptions differ, and generate an HTML-safe reference table.

**B. UI normalization.** Extend Problem 3 with Unicode normalization. Decide whether `ß` and `ss` should be treated alike and make the decision part of your tests.

**C. Partial CSV success.** Extend Problem 9 to produce a generator of `(Shipment09 | None, CsvIssue09 | None)` pairs and use it on a large artificial input without materializing the full input list.

**D. Database upgrade.** Extend Problem 10 with a transaction that repairs legacy rows and verifies no legacy values remain before the migration is committed.

**E. Compatibility versioning.** Add an explicit version byte to Problem 16's binary format and document how old and new decoders respond to unknown versions and new metric codes.

**Learning tip:** For every new exercise, write the invariant and at least one malformed-input test *before* expanding the implementation.